# PREM canonical generator
Generates pointwise mass density plus an explicitly labelled composition extension for neutrino matter potentials.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT=Path.cwd().resolve().parents[3]; raw=ROOT/'data/earth/prem/raw/prem_table_full.txt'
rows=[]
for line in raw.read_text().splitlines()[2:]:
    f=line.split()
    if len(f)==12:
        try: rows.append([float(value) if value != '???' else float('nan') for value in f])
        except ValueError: pass
t=pd.DataFrame(rows,columns=['radius_km','mass_density_g_cm3','vp_km_s','vs_km_s','phi_km2_s2','bulk_modulus_GPa','lambda_GPa','mu_GPa','poisson_ratio','gravity_m_s2','pressure_GPa','depth_km'])
t['radius_fraction']=t.radius_km/6371.0
def ye(row):
    if row.mass_density_g_cm3<1.1 and row.depth_km<3: return 0.5556
    return 0.467 if row.depth_km>2891 else 0.495
t['electron_fraction']=t.apply(ye,axis=1)
t['electron_density_mol_cm3']=t.mass_density_g_cm3*t.electron_fraction
t['neutron_density_mol_cm3']=t.mass_density_g_cm3*(1-t.electron_fraction)
out=ROOT/'data/earth/prem/density/prem_density.csv'; t.sort_values('radius_km').to_csv(out,index=False)
print(out,t.shape); display(t.head())

G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts\data\earth\prem\density\prem_density.csv (80, 16)


,radius_km,mass_density_g_cm3,vp_km_s,vs_km_s,phi_km2_s2,bulk_modulus_GPa,lambda_GPa,mu_GPa,poisson_ratio,gravity_m_s2,pressure_GPa,depth_km,radius_fraction,electron_fraction,electron_density_mol_cm3,neutron_density_mol_cm3
0,6371.0,1.02,1.45,0.0,2.10,2.1,2.1,0.0,0.5000,9.822,0.000,0.0,1.000000,0.5556,0.566712,0.453288
1,6368.0,1.02,1.45,0.0,2.10,2.1,2.1,0.0,0.5000,9.828,0.030,3.0,0.999529,0.4950,0.504900,0.515100
2,6368.0,2.60,5.80,3.2,19.99,52.0,34.2,26.6,0.2812,9.828,0.030,3.0,0.999529,0.4950,1.287000,1.313000
3,6356.0,2.60,5.80,3.2,19.99,52.0,34.2,26.6,0.2812,9.839,0.337,15.0,0.997646,0.4950,1.287000,1.313000
4,6356.0,2.90,6.80,3.9,25.96,75.3,45.9,44.1,0.2549,9.839,0.337,15.0,0.997646,0.4950,1.435500,1.464500


## 3. High-resolution radial table

PREM is piecewise polynomial and contains physical discontinuities at layer boundaries. The parallel table below resamples each continuous branch independently at 1 km spacing and retains both sides of every discontinuity. It is a numerical convenience, not an independent or more accurate measurement than the source PREM parametrisation.


In [ ]:
# Dense 1-km sampling without interpolating across PREM discontinuities.
def continuous_branches(table):
    start = 0
    for i in range(len(table) - 1):
        if table.radius_km.iloc[i + 1] == table.radius_km.iloc[i]:
            yield table.iloc[start:i + 1]
            start = i + 1
    yield table.iloc[start:]

dense_parts = []
for branch in continuous_branches(t.sort_values("radius_km", kind="stable").reset_index(drop=True)):
    r0, r1 = branch.radius_km.iloc[[0, -1]]
    grid = np.unique(np.r_[r0, np.arange(np.ceil(r0), np.floor(r1) + 1.0, 1.0), r1])
    dense = pd.DataFrame({"radius_km": grid})
    for column in branch.columns.drop("radius_km"):
        valid = branch[["radius_km", column]].dropna()
        dense[column] = (
            np.interp(grid, valid.radius_km, valid[column])
            if len(valid) >= 2 else valid[column].iloc[0] if len(valid) else np.nan
        )
    dense_parts.append(dense)
prem_dense = pd.concat(dense_parts, ignore_index=True)
dense_out = ROOT / "data/earth/prem/density/prem_density_dense.csv"
prem_dense.to_csv(dense_out, index=False)
print(dense_out, prem_dense.shape)
display(prem_dense.head())
